# Create and update columns · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 08. Create and update columns

Calculate new fields, tidy column names, and change selected values safely.

Before you start: Know arithmetic, column selection, Boolean masks, and assignment.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/05_add_columns.html

### M08C01. Calculate a new column

Assign a Series to a new column name to add a field. Arithmetic across columns works row by row after index alignment. Here the artificial Score_Per_Hour ratio demonstrates division; it is not a validated measure of student ability. Always consider units and the meaning of a calculation.

**Read:** Which part of the assignment names the new column?

Your explanation: _write here_.

**Predict:** What is Amira's Score_Per_Hour before rounding?

Your explanation: _write here_.

**Explain:** Why should a high ratio not be treated as proof of better learning?

Your explanation: _write here_.



In [ ]:
# Predict before running: What is Amira's Score_Per_Hour before rounding?
df = students()
df["Score_Per_Hour"] = df["Score"] / df["Study_Hours"]
result = df[["Name", "Score", "Study_Hours", "Score_Per_Hour"]]
print(result.head(3).round(2))

**Your turn:** Start with a fresh df = students(). Create Score_Fraction by dividing Score by 100. Store Name, Score, and Score_Fraction for all students in result, then print its first three rows.

For the check, keep these result names: `df`, `result`.

**Hint:** Assign the division result to df["Score_Fraction"].

**Watch for this:** Before dividing by a column, check for zero and missing denominators; they can create undefined results.

In [ ]:
# Change this code to complete the task.
# Question: Start with a fresh df = students(). Create Score_Fraction by dividing Score by 100. Store Name, Score, and Score_Fraction for all students in result, then print its first three rows.
df = students()
df["Score_Per_Hour"] = df["Score"] / df["Study_Hours"]
result = df[["Name", "Score", "Study_Hours", "Score_Per_Hour"]]
print(result.head(3).round(2))

**Quick check:** What happens when you assign to a column name that does not yet exist?

1. Pandas adds the column.
2. Pandas always deletes a row.
3. Pandas sorts the index.

**Explain what you learned:** I can create a calculated column and state its units and limits.

### M08C02. Use assign, rename, and drop

assign() returns a DataFrame with new or replaced columns. rename(columns=...) changes column labels, while drop(columns=...) removes named columns. These methods return new objects by default. Assign the returned result so your intended changes are kept.

**Read:** What is stored in the new Passed column?

Your explanation: _write here_.

**Predict:** Does result still have a column named Name?

Your explanation: _write here_.

**Explain:** Why does the final table still have nine columns?

Your explanation: _write here_.



In [ ]:
# Predict before running: Does result still have a column named Name?
df = students()
result = (df.assign(Passed=df["Score"] >= 60)
            .rename(columns={"Name": "Student_Name"})
            .drop(columns=["Join_Date"]))
print(result[["Student_Name", "Score", "Passed"]].head(4))
print("Result shape:", result.shape)

**Your turn:** Use assign to create Needs_Support for scores below 70. Rename Attendance to Attendance_Percent and drop Join_Date. Keep the result in result.

For the check, keep these result names: `df`, `result`.

**Hint:** Provide a Boolean Series to assign, a mapping to rename, and a list of names to drop.

**Watch for this:** Calling df.drop(...) without assigning its returned result does not change df by default.

In [ ]:
# Change this code to complete the task.
# Question: Use assign to create Needs_Support for scores below 70. Rename Attendance to Attendance_Percent and drop Join_Date. Keep the result in result.
df = students()
result = (df.assign(Passed=df["Score"] >= 60)
            .rename(columns={"Name": "Student_Name"})
            .drop(columns=["Join_Date"]))
print(result[["Student_Name", "Score", "Passed"]].head(4))
print("Result shape:", result.shape)

**Quick check:** Which removes a named column from the returned table?

1. df.rename(columns={"Score": "Mark"})
2. df.drop(columns=["Score"])
3. df.assign(Score=0)

**Explain what you learned:** I can add fields and tidy column structure while keeping the original available.

### M08C03. Update the original with one loc assignment

Use one .loc[row_condition, column_name] assignment to change selected values in a DataFrame. This explicitly identifies both the rows and the field being changed. Avoid chained assignment such as df[mask]["Score"] = value; it does not reliably update the intended original object and does not do so under pandas 3 Copy-on-Write.

**Read:** Which row and column are changed?

Your explanation: _write here_.

**Predict:** What is Dana's score after the assignment?

Your explanation: _write here_.

**Explain:** Why does a single loc assignment state your intention more clearly than two chained brackets?

Your explanation: _write here_.



In [ ]:
# Predict before running: What is Dana's score after the assignment?
df = students()
mask = df["Student_ID"] == "S004"
df.loc[mask, "Score"] = 67
result = df.loc[mask, ["Student_ID", "Name", "Score"]]
print(result)

**Your turn:** Apply a practice bonus of 3 points to students with Attendance at least 95. Cap scores at 100 and show only the updated students.

For the check, keep these result names: `df`, `result`.

**Hint:** Use one .loc[mask, "Score"] assignment; calculate from the same selected Score values and clip at 100.

**Watch for this:** Repeatedly running an incremental update on the same table would repeat the bonus; this cell starts with a fresh table each time.

In [ ]:
# Change this code to complete the task.
# Question: Apply a practice bonus of 3 points to students with Attendance at least 95. Cap scores at 100 and show only the updated students.
df = students()
mask = df["Student_ID"] == "S004"
df.loc[mask, "Score"] = 67
result = df.loc[mask, ["Student_ID", "Name", "Score"]]
print(result)

**Quick check:** Which is the clear way to update selected scores in df?

1. df[mask]["Score"] = 67
2. df["Score"][mask] = 67
3. df.loc[mask, "Score"] = 67

**Explain what you learned:** I can modify selected values using one explicit loc assignment.

### M08C04. Keep an independent working copy

Assigning original to another name creates an alias: both names refer to the same Python object. copy() creates a separate DataFrame for this ordinary scalar-valued teaching table. Use a fresh copy when you want to experiment without changing the original. Pandas copy rules do not replace the need to understand which object you are editing.

**Read:** What does .copy() let us do in this example?

Your explanation: _write here_.

**Predict:** What two first-score values are printed?

Your explanation: _write here_.

**Explain:** What would change if the code used working = original instead?

Your explanation: _write here_.



In [ ]:
# Predict before running: What two first-score values are printed?
original = students()
working = original.copy()
working.loc[0, "Score"] = 0
print("Original first score:", original.loc[0, "Score"])
print("Working first score:", working.loc[0, "Score"])
print("Same object:", working is original)

**Your turn:** Copy only the RAK rows into working. Add 1 to their scores in the copy and verify Amira's original score is still 88.

For the check, keep these result names: `original`, `working`.

**Hint:** Filter with loc, call .copy(), and update the selected table rather than original.

**Watch for this:** For columns containing nested mutable Python objects, even a deep DataFrame copy does not recursively copy those inner objects; this table uses ordinary scalar values.

In [ ]:
# Change this code to complete the task.
# Question: Copy only the RAK rows into working. Add 1 to their scores in the copy and verify Amira's original score is still 88.
original = students()
working = original.copy()
working.loc[0, "Score"] = 0
print("Original first score:", original.loc[0, "Score"])
print("Working first score:", working.loc[0, "Score"])
print("Same object:", working is original)

**Quick check:** What does working = original do?

1. Makes two names refer to the same object.
2. Always creates a separate deep copy.
3. Exports the data to a file.

**Explain what you learned:** I can protect source values while testing changes on a separate table.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates